# 第36章 数据结构与主题

理解Seaborn的长表映射、Axes级与Figure级接口、主题和调色板。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

使用DataFrame直接完成统计聚合、分类映射和统一视觉风格。

## 数据结构

优先使用每行一个观察、每列一个变量的长表。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 sns.axes_style("ticks") 改为 "whitegrid" 或 "dark"，对比不同主题风格
2. 修改 palette 参数从 "Set2" 为 "pastel" 或 "muted"，观察调色板变化
3. 在 scatterplot 中添加 style="channel" 参数，观察形状映射与颜色映射的组合效果


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid", context="notebook")
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category = diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value = diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel = taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend = taxis["tip"], sales=taxis["total"],
    conversion = (taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date = pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region = "AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.scatterplot(data=marketing, x="visits", y="sales", hue="channel", ax=ax)
ax.set(title="Seaborn长表映射", xlabel="访问量", ylabel="销售额")
ax.legend(title="渠道", frameon=False)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
with sns.axes_style("ticks"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.countplot(data=orders, x="category", color="#1a73e8", ax=axes[0])
    sns.boxplot(data=orders, x="category", y="order_value", hue="category", palette="Set2", legend=False, ax=axes[1])
    axes[0].set(title="订单量", xlabel="品类", ylabel="订单数")
    axes[1].set(title="客单价分布", xlabel="品类", ylabel="元")
    sns.despine()
    fig.tight_layout()
plt.show()


## 3. 参数说明

- data：数据表
- x/y：位置变量
- hue：颜色分组
- style/size：其他映射


## 4. 结果解读

先确认每个视觉通道对应哪一列，再判断函数是否自动执行了统计聚合。


## 常见误区

- 宽表和长表混用
- 不知道barplot默认计算均值
- Figure级函数传入已有Axes


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
sns.set_theme(style="whitegrid", palette="colorblind")
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=orders, x="category", y="order_value", errorbar=None, ax=ax)
ax.set(title="品类平均客单价", xlabel="品类", ylabel="元")
fig.tight_layout()
plt.show()


## 本章小结

理解Seaborn的长表映射、Axes级与Figure级接口、主题和调色板。


### 你已经掌握

- 判断数据结构与主题的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 使用DataFrame直接完成统计聚合、分类映射和统一视觉风格。 |
| 数据结构 | 优先使用每行一个观察、每列一个变量的长表。 |
| 结果解读 | 先确认每个视觉通道对应哪一列，再判断函数是否自动执行了统计聚合。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `data` | 数据表 |
| `x/y` | 位置变量 |
| `hue` | 颜色分组 |
| `style/size` | 其他映射 |


### 需要注意

- 宽表和长表混用
- 不知道barplot默认计算均值
- Figure级函数传入已有Axes


### 完成检查

- [ ] 能判断什么问题适合使用数据结构与主题
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
